# 1. Adding project folder to the system path

In [6]:
from PIL import Image
from pathlib import Path
import torchvision.transforms as T
import os
import sys
#===========================================
# Path to dataset
#===========================================
project_path = Path('/home/anhnam/Data/VideoAnomaly')
ped2_dataset_folder = Path('/home/dataset/ped2')
train_dir = ped2_dataset_folder / 'training'
test_dir = ped2_dataset_folder / 'testing'

if train_dir.is_dir():
    print(f"{train_dir} directory exists.")
    
# adding project folder to the system path
#sys.path.insert(0, '/home/anhnam/Data/VAD')

#%pwd # the dir which contains this notebook file

/home/dataset/ped2/training directory exists.


# 2. PIL image to Tensor

In [10]:
def get_transform_PIL_Tensor(size, 
                  method=T.InterpolationMode.BICUBIC, 
                  is_toTensor=True,
                  is_grayScale=False,
                  is_normalize=True):
    """
    Returns a transform (ToTensor(), Normalize()) 
    which convert a image from [0,255] to [-1,1]
    """
    w, h = size
    #new_size = [make_power_2(w), make_power_2(h)]
    new_size = [w, h]

    transform_list = [T.Resize(new_size, method)]
    if is_grayScale:
        transform_list += [T.Grayscale()]
    if is_toTensor:# Scales data into [0,1]
        transform_list += [T.ToTensor()] 
    if is_normalize:# Scale between [-1, 1] 
        if is_grayScale:
            transform_list += [T.Normalize(mean=(0.5),
                                       std=(0.5))]
        else:
            transform_list += [T.Normalize(mean=(0.5, 0.5, 0.5),
                                       std=(0.5, 0.5, 0.5))]
    return T.Compose(transform_list)

In [12]:
image_path = train_dir / '01/000.jpg'
PIL_image = Image.open(image_path).convert('RGB') 

new_size=[4,4]
transform = get_transform_PIL_Tensor(size=new_size, is_toTensor=True,
                          is_grayScale=False, is_normalize=True)
image_tensor = transform(PIL_image)

print('image_tensor.shape = ', image_tensor.shape)
print(image_tensor) # [-1,1]


image_tensor.shape =  torch.Size([3, 4, 4])
tensor([[[-0.5059, -0.4745, -0.3961, -0.3725],
         [-0.2941, -0.3569, -0.3647, -0.3882],
         [ 0.0510,  0.1137,  0.1765,  0.1686],
         [ 0.2000,  0.3569,  0.3255,  0.1216]],

        [[-0.5059, -0.4745, -0.3961, -0.3725],
         [-0.2941, -0.3569, -0.3647, -0.3882],
         [ 0.0510,  0.1137,  0.1765,  0.1686],
         [ 0.2000,  0.3569,  0.3255,  0.1216]],

        [[-0.5059, -0.4745, -0.3961, -0.3725],
         [-0.2941, -0.3569, -0.3647, -0.3882],
         [ 0.0510,  0.1137,  0.1765,  0.1686],
         [ 0.2000,  0.3569,  0.3255,  0.1216]]])


In [6]:
transform = get_transform_PIL_Tensor(size=new_size, is_toTensor=True,
                          is_grayScale=False, is_normalize=False)
image_tensor = transform(PIL_image)

print('image_tensor.shape = ', image_tensor.shape)
print(image_tensor) # [0,1]

tensor([[[0.2627, 0.2745, 0.2941],
         [0.4353, 0.4392, 0.4471],
         [0.6118, 0.6706, 0.6039]],

        [[0.2627, 0.2745, 0.2941],
         [0.4353, 0.4392, 0.4471],
         [0.6118, 0.6706, 0.6039]],

        [[0.2627, 0.2745, 0.2941],
         [0.4353, 0.4392, 0.4471],
         [0.6118, 0.6706, 0.6039]]])


In [7]:
transform = get_transform_PIL_Tensor(size=new_size, is_toTensor=False,
                          is_grayScale=False, is_normalize=False)
image_tensor = transform(PIL_image)

print('image_tensor.shape = ', image_tensor.shape)
print(image_tensor) # [0,1]

<PIL.Image.Image image mode=RGB size=3x3 at 0x7F1D3152E910>


# 3. CV2 to Tensor

In [25]:
from cv2_lib import *
import torchvision.transforms as transforms
import torch

image_path = train_dir / '01/000.jpg'
image_path2 = train_dir / '01/000.jpg'

print('image_path = ', image_path)
CV2_image = CV2_load_image(str(image_path),1, 4, 6)
CV2_image2 = CV2_load_image(str(image_path2),1, 4, 6)

transform = transforms.Compose([transforms.ToTensor()])

image_tensor = transform(CV2_image)
image_tensor2 = transform(CV2_image2)
print('image_tensor.shape1 = ', image_tensor.shape) # torch.Size([3, 4, 6]) (C, H, W)
print(image_tensor) # [-1,1]

image_tensor = torch.unsqueeze(image_tensor, dim=0) 
print('image_tensor.shape2 = ', image_tensor.shape) # torch.Size([1, 3, 4, 6])

image_tensor2 = torch.unsqueeze(image_tensor2, dim=0) # torch.Size([1, 3, 4, 6])
aug_box = []
aug_box.append(image_tensor)
aug_box.append(image_tensor2)
result = torch.cat(aug_box, dim=0)

print('result.shape2 = ', result.shape) # torch.Size([2, 3, 4, 6]) # (N, C, H, W)
result = result.transpose(0,1)
print('result.shape2 = ', result.shape) # torch.Size([3, 2, 4, 6])  # (C, N, H, W)

image_path =  /home/dataset/ped2/training/01/000.jpg
image_tensor.shape1 =  torch.Size([3, 4, 6])
tensor([[[-0.5843, -0.3725, -0.4902, -0.2235, -0.5216, -0.4745],
         [-0.4431, -0.4745, -0.3804, -0.3569, -0.3804, -0.4039],
         [ 0.1137, -0.4275,  0.2471,  0.3490,  0.1922,  0.2941],
         [ 0.3020,  0.3725,  0.3255,  0.3412,  0.3412,  0.3412]],

        [[-0.5843, -0.3725, -0.4902, -0.2235, -0.5216, -0.4745],
         [-0.4431, -0.4745, -0.3804, -0.3569, -0.3804, -0.4039],
         [ 0.1137, -0.4275,  0.2471,  0.3490,  0.1922,  0.2941],
         [ 0.3020,  0.3725,  0.3255,  0.3412,  0.3412,  0.3412]],

        [[-0.5843, -0.3725, -0.4902, -0.2235, -0.5216, -0.4745],
         [-0.4431, -0.4745, -0.3804, -0.3569, -0.3804, -0.4039],
         [ 0.1137, -0.4275,  0.2471,  0.3490,  0.1922,  0.2941],
         [ 0.3020,  0.3725,  0.3255,  0.3412,  0.3412,  0.3412]]])
image_tensor.shape2 =  torch.Size([1, 3, 4, 6])
result.shape2 =  torch.Size([2, 3, 4, 6])
result.shape2 =  torch.Siz